# Exploración y limpieza de datos — Inside Airbnb (CDMX)

**Fuente:** Inside Airbnb, Ciudad de México · snapshot 2026-06-15 (`listings.csv.gz`).

**Objetivo:** cargar el dataset, explorarlo y dejar `df_ml` limpio y confiable para entrenar el modelo de precio.


In [ ]:
import pandas as pd

# Opciones de despliegue (aplican a todo el notebook).
pd.set_option("display.max_columns", None)   # muestra todas las columnas
pd.set_option("display.max_rows", 100)        # hasta 100 filas
pd.set_option("display.max_colwidth", 60)      # recorta textos largos (description/amenities)

# pandas lee el .gz directo. El notebook está en notebooks/ y los datos en ../data/
df = pd.read_csv("../data/listings.csv.gz")
print("Filas, columnas:", df.shape)


## 1. Vistazo inicial

Primeras filas y resumen de tipos / nulos por columna.


In [ ]:
df.head()


In [ ]:
df.info()


## 2. Vista cómoda — solo columnas relevantes

Mostramos únicamente las columnas que usaremos (features + target) más unas de referencia para reconocer cada propiedad.

In [ ]:
mostrar = [
    # --- Referencia (para ubicar la propiedad; NO van al modelo) ---
    "listing_url", "name", "description", "host_name",
    # --- Target (lo que predecimos) ---
    "price",
    # --- Features de la propiedad ---
    "property_type", "room_type", "accommodates", "bathrooms", "bedrooms", "beds", "amenities",
    # --- Features de ubicación ---
    "neighbourhood_cleansed", "latitude", "longitude",
    # --- Solo para filtrar (no son features) ---
    "review_scores_rating", "number_of_reviews", "number_of_reviews_ltm",
]

df[mostrar].head(30)


## 3. Exploración de precio y outliers

Antes de limpiar, revisamos los precios extremos para justificar los cortes. `price` viene como texto (`"$2,066.00"`), así que lo convertimos a número **solo para explorar** (no modifica `df`).


In [ ]:
# Precio a número solo para explorar (no modifica df).
precio = (df["price"].str.replace("$", "", regex=False)
                     .str.replace(",", "", regex=False).astype(float))
expl = df.assign(price_num=precio).dropna(subset=["price_num"]) #copia nueva, y eliminamos anuncios sin precio
#Un anuncio puedo no tener precio por ser nuevo y no haber precio base, o por anfitriones que han pausado su anuncio o mas de un año de inactividad

#checamos los percentiles que coincidan con la realidad de mercado
print("Cortes:", expl["price_num"].quantile([0.01, 0.05, 0.95, 0.99]).round(1).to_dict()) 


print("\n--- 12 más caras ---")
print(expl.nlargest(12, "price_num")[
    ["price_num", "room_type", "accommodates", "bedrooms", "neighbourhood_cleansed"]
])

print("\n--- 8 más baratas ---")
print(expl.nsmallest(8, "price_num")[
    ["price_num", "room_type", "accommodates", "bedrooms", "neighbourhood_cleansed"]
])


## 4. Limpieza final consolidada

Toda la limpieza en un solo paso reproducible:
1. `price` a número y fuera filas sin precio (es el target).
2. **Filtro de confianza:** solo propiedades bien calificadas (≥ 4.7), con historial (≥ 5 reseñas) y activas el último año. Las reseñas se usan **solo para filtrar**, no como features.
3. **Recorte de outliers** de precio (1 % de cada cola) sobre las ya confiables.
4. Quitamos las columnas que solo servían de filtro.


In [ ]:
# ========== LIMPIEZA FINAL CONSOLIDADA ==========
RATING_MIN  = 4.7   # estándar real de Airbnb (4.5 es bajo en su escala)
MIN_RESENAS = 5

cols = [
    "price",
    "room_type", "property_type",
    "accommodates", "bedrooms", "bathrooms", "beds",
    "neighbourhood_cleansed", "latitude", "longitude",
    "amenities",
    "review_scores_rating", "number_of_reviews", "number_of_reviews_ltm",  # SOLO para filtrar
]
d = df[cols].copy()

# 1. price: texto -> número, y fuera filas sin precio (es el target).
d["price"] = (d["price"].str.replace("$", "", regex=False)
                        .str.replace(",", "", regex=False).astype(float))
d = d.dropna(subset=["price"])
print("Filas Con precio:          ", len(d))

# 2. Filtro de confianza: bien calificada + con historial + activa este año.
d = d[(d["review_scores_rating"] >= RATING_MIN) &
      (d["number_of_reviews"] >= MIN_RESENAS) &   # si no se cumplen las 3 se elimina la fila
      (d["number_of_reviews_ltm"] > 0)]
print("Filas con filtro calidad: ", len(d))

# 3. Recorte de outliers de precio (1% de cada cola) sobre las ya confiables.
bajo, alto = d["price"].quantile([0.01, 0.99])
d = d[(d["price"] >= bajo) & (d["price"] <= alto)]
print(f"Filas con recorte precio:  {len(d)}  (rango {bajo:.0f}-{alto:.0f})") # RANGO DE PRECIOS despued de aplicar el percentil

# 4. Quitamos las columnas que solo servían de filtro (no son features).
df_ml = d.drop(columns=["review_scores_rating", "number_of_reviews", "number_of_reviews_ltm"])

print("\nColumnas finales:", list(df_ml.columns))
df_ml["price"].describe()


### Verificación: los outliers de precio sobreviven al filtro de calidad

Aunque una propiedad tenga buenas reseñas, su precio puede traer un error de captura. Tras el filtro de calidad (sin recorte), el max sigue absurdo, por eso el recorte del 1% sí hace falta.

In [ ]:
precio = (df["price"].str.replace("$", "", regex=False)
                     .str.replace(",", "", regex=False).astype(float))
q = df.assign(price_num=precio).dropna(subset=["price_num"])
q = q[(q["review_scores_rating"] >= 4.7) &
      (q["number_of_reviews"] >= 5) &
      (q["number_of_reviews_ltm"] > 0)]

print("Tras filtro calidad (sin recorte):", len(q))
print(q["price_num"].describe())

print("\n--- 10 más caras ---")
print(q.nlargest(10, "price_num")[
    ["price_num", "room_type", "accommodates", "bedrooms", "neighbourhood_cleansed"]
])

### Rastreo de los monstruos: precios absurdos y su enlace

Listamos las propiedades con precio mayor o igual a 100,000 por noche junto a su link en Airbnb, para verificar a mano que son errores de captura y no lujo real.

In [ ]:
precio = (df["price"].str.replace("$", "", regex=False)
                     .str.replace(",", "", regex=False).astype(float))
tmp = df.assign(price_num=precio)

# Todas las propiedades con precio >= 100,000 basura.
monstruos = tmp[tmp["price_num"] >= 100000].sort_values("price_num", ascending=False)
print("Monstruos (precio >= 100,000):", len(monstruos), "\n")

for _, r in monstruos.head(10).iterrows():
    print(f"${r.price_num:,.0f}/noche | {r.room_type}, {r.accommodates} huésp. | {r.neighbourhood_cleansed}")
    print(f"   {r['name']}")
    print(f"   {r.listing_url}")
    print()

## 5. Resultado — `df_ml` listo

`df_ml` tiene ~14,500 propiedades confiables de CDMX, con 10 features + `price` (target).

**Siguiente paso:** manejar los nulos de `bedrooms` / `bathrooms` / `beds` y entrenar el Random Forest.


## 6. Manejo de nulos

Un modelo no puede entrenar con celdas vacías. Revisamos cuántos nulos quedan en `df_ml` y decidimos si rellenarlos (con la mediana) o descartarlos.

In [ ]:
# Cuantos nulos (celdas vacias) quedan por columna en df_ml.
df_ml.isna().sum()

In [ ]:
# Rellenamos los nulos de bedrooms/bathrooms/beds con su mediana. para no perder propiedades
# (Borrar esas filas costaria ~16% de los datos por bedrooms; mejor imputar.)
for col in ["bedrooms", "bathrooms", "beds"]:
    mediana = df_ml[col].median()
    df_ml[col] = df_ml[col].fillna(mediana)
    print(f"{col}: relleno con mediana = {mediana}")

# Verificamos que ya no queden nulos en todo df_ml.
print("\nNulos restantes:", df_ml.isna().sum().sum())

In [ ]:
# Rastreamos propiedades SIN bedrooms/bathrooms/beds, con su link para revisar.
precio = (df["price"].str.replace("$", "", regex=False)
                     .str.replace(",", "", regex=False).astype(float))
tmp = df.assign(price_num=precio)

# Nos quedamos con las confiables (mismo filtro de calidad) que les falte ALGUN dato.
q = tmp[(tmp["review_scores_rating"] >= 4.7) &
        (tmp["number_of_reviews"] >= 5) &
        (tmp["number_of_reviews_ltm"] > 0)]
sin_datos = q[q["bedrooms"].isna() | q["bathrooms"].isna() | q["beds"].isna()]

print("Confiables a las que les falta bedrooms/bathrooms/beds:", len(sin_datos), "\n")

for _, r in sin_datos.head(10).iterrows():
    print(f"{r.room_type}, {r.accommodates} huesp | rec={r.bedrooms}, banos={r.bathrooms}, camas={r.beds}")
    print(f"   {r['name']}")
    print(f"   {r.listing_url}")
    print()

## 7. Encoding (texto a numeros)

El modelo solo entiende numeros. Convertimos las columnas de texto a numericas:
- `room_type`, `property_type`, `neighbourhood_cleansed` -> one-hot (una columna binaria por categoria).
- `amenities` -> aparte, porque es una lista de varias cosas.

Primero vemos cuantas categorias distintas tiene cada una (one-hot crea una columna por categoria).

In [ ]:
# Cuantas categorias distintas tiene cada columna de texto (antes de codificar).
for col in ["room_type", "property_type", "neighbourhood_cleansed"]:
    print(f"{col}: {df_ml[col].nunique()} categorias")
    print("   top 3:", df_ml[col].value_counts().head(3).index.tolist())
    print()